In [0]:
from pyspark.sql import functions as F

FILES = "/Volumes/workspace/bronze/landing/files"
source = "elexon_system_prices"

raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{FILES}/{source}/*/*Z_????????.json")
    .select(
        F.explode("data").alias("r"),
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .select("r.*", "_source_file")
    .withColumn("_partition_day",
                F.to_date(F.regexp_extract("_source_file", r"/(\d{4}-\d{2}-\d{2})/", 1)))
    .withColumn("_fetched_at",
                F.to_timestamp(F.regexp_extract("_source_file", r"/(\d{8}T\d{12})Z_", 1),
                               "yyyyMMdd'T'HHmmssSSSSSS"))
)

raw.printSchema()
print("rows:", raw.count(), "| days:", raw.select("_partition_day").distinct().count())
display(raw.limit(5))

root
 |-- bsadDefaulted: boolean (nullable = true)
 |-- buyPriceAdjustment: double (nullable = true)
 |-- createdDateTime: string (nullable = true)
 |-- netImbalanceVolume: double (nullable = true)
 |-- priceDerivationCode: string (nullable = true)
 |-- replacementPrice: double (nullable = true)
 |-- replacementPriceReferenceVolume: double (nullable = true)
 |-- reserveScarcityPrice: double (nullable = true)
 |-- sellPriceAdjustment: double (nullable = true)
 |-- settlementDate: string (nullable = true)
 |-- settlementPeriod: long (nullable = true)
 |-- startTime: string (nullable = true)
 |-- systemBuyPrice: double (nullable = true)
 |-- systemSellPrice: double (nullable = true)
 |-- totalAcceptedBidVolume: double (nullable = true)
 |-- totalAcceptedOfferVolume: double (nullable = true)
 |-- totalAdjustmentBuyVolume: double (nullable = true)
 |-- totalAdjustmentSellVolume: double (nullable = true)
 |-- totalSystemTaggedAcceptedBidVolume: double (nullable = true)
 |-- totalSystemTagged

bsadDefaulted,buyPriceAdjustment,createdDateTime,netImbalanceVolume,priceDerivationCode,replacementPrice,replacementPriceReferenceVolume,reserveScarcityPrice,sellPriceAdjustment,settlementDate,settlementPeriod,startTime,systemBuyPrice,systemSellPrice,totalAcceptedBidVolume,totalAcceptedOfferVolume,totalAdjustmentBuyVolume,totalAdjustmentSellVolume,totalSystemTaggedAcceptedBidVolume,totalSystemTaggedAcceptedOfferVolume,totalSystemTaggedAdjustmentBuyVolume,totalSystemTaggedAdjustmentSellVolume,_source_file,_partition_day,_fetched_at
false,0.0,2024-10-27T23:44:36Z,-58.48727855886058,N,null,null,0.0,0.0,2024-10-27,1,2024-10-26T23:00:00Z,55.0,55.0,-1350.4876745215236,1024.4424081078062,267.35,0.0,-1349.4876745215236,1024.4424081078062,267.35,null,dbfs:/Volumes/workspace/bronze/landing/files/elexon_system_prices/2024-10-27/20261013T230014411856Z_d37f6c0c.json,2024-10-27,2026-10-13T23:00:14.411Z
false,0.0,2024-10-28T00:14:35Z,-243.04498387626262,N,null,null,0.0,0.0,2024-10-27,2,2024-10-26T23:30:00Z,0.5,0.5,-1397.3492768827161,929.2855098737374,225.0,0.0,-1396.3492768827161,929.2855098737374,225.0,null,dbfs:/Volumes/workspace/bronze/landing/files/elexon_system_prices/2024-10-27/20261013T230014411856Z_d37f6c0c.json,2024-10-27,2026-10-13T23:00:14.411Z
false,0.0,2024-10-28T00:44:31Z,-359.9708436111111,N,null,null,0.0,0.0,2024-10-27,3,2024-10-27T00:00:00Z,51.02,51.02,-1194.5861213888888,1359.611111111111,225.0,-750.0,-1194.5861213888888,1359.611111111111,225.0,-749.0,dbfs:/Volumes/workspace/bronze/landing/files/elexon_system_prices/2024-10-27/20261013T230014411856Z_d37f6c0c.json,2024-10-27,2026-10-13T23:00:14.411Z
false,0.0,2024-10-28T01:14:35Z,-518.9392379166667,N,null,null,0.0,0.0,2024-10-27,4,2024-10-27T00:30:00Z,48.59018867924527,48.59018867924527,-1326.7670833333334,1332.8278454166666,225.0,-750.0,-1326.7670833333334,1332.8278454166666,225.0,-749.0,dbfs:/Volumes/workspace/bronze/landing/files/elexon_system_prices/2024-10-27/20261013T230014411856Z_d37f6c0c.json,2024-10-27,2026-10-13T23:00:14.411Z
false,0.0,2024-10-28T01:44:40Z,-527.8541666666666,N,null,null,0.0,0.0,2024-10-27,5,2024-10-27T01:00:00Z,46.08666666666666,46.08666666666666,-1349.3166666666666,1496.4583333333333,225.0,-900.0,-1349.3166666666666,1496.4583333333333,225.0,-899.0,dbfs:/Volumes/workspace/bronze/landing/files/elexon_system_prices/2024-10-27/20261013T230014411856Z_d37f6c0c.json,2024-10-27,2026-10-13T23:00:14.411Z


In [0]:
per_day = raw.groupBy("_partition_day").count()
display(per_day.filter("count != 48").orderBy("_partition_day"))

_partition_day,count
2024-10-27,50
2025-03-30,46
2025-10-26,50
2026-03-29,46
2026-10-13,47


In [0]:
bronze = raw.withColumn("_ingested_at", F.current_timestamp())

(bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("workspace.bronze.elexon_system_prices"))

print(spark.table("workspace.bronze.elexon_system_prices").count())
display(spark.sql("DESCRIBE HISTORY workspace.bronze.elexon_system_prices")
        .select("version", "timestamp", "operation", "operationMetrics"))

35663


version,timestamp,operation,operationMetrics
0,2026-10-16T00:25:05.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 35663, numOutputBytes -> 1883203)"


In [0]:
DATA_LIST_SOURCES = [
    "elexon_system_prices",
    "elexon_settled_prices",
    "elexon_demand_outturn",
    "elexon_generation_by_fuel",
    "elexon_wind_solar_actual",
    "elexon_wind_solar_day_ahead",
    "elexon_demand_forecast",
    "elexon_wind_forecast",
    "carbon_intensity",
]


def read_data_list_source(source: str):
    """Read every landed file of a source whose JSON is {"data": [records]}."""
    return (
        spark.read
        .option("multiLine", True)
        .json(f"{FILES}/{source}/*/*Z_????????.json")
        .select(F.explode("data").alias("r"),
                F.col("_metadata.file_path").alias("_source_file"))
        .select("r.*", "_source_file")
        .withColumn("_partition_day",
                    F.to_date(F.regexp_extract("_source_file", r"/(\d{4}-\d{2}-\d{2})/", 1)))
        .withColumn("_fetched_at",
                    F.to_timestamp(F.regexp_extract("_source_file", r"/(\d{8}T\d{12})Z_", 1),
                                   "yyyyMMdd'T'HHmmssSSSSSS"))
        .withColumn("_ingested_at", F.current_timestamp())
    )


summary = []
for source in DATA_LIST_SOURCES:
    df = read_data_list_source(source)
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", True)
       .saveAsTable(f"workspace.bronze.{source}"))
    t = spark.table(f"workspace.bronze.{source}")
    summary.append((source, t.count(), t.select("_partition_day").distinct().count(),
                    t.select("_source_file").distinct().count()))
    print("done:", source)

display(spark.createDataFrame(summary, "source string, rows long, days long, files long"))

done: elexon_system_prices
done: elexon_settled_prices
done: elexon_demand_outturn
done: elexon_generation_by_fuel
done: elexon_wind_solar_actual
done: elexon_wind_solar_day_ahead


In [0]:
src = "elexon_wind_solar_day_ahead"
on_disk = {f.name.rstrip("/") for f in dbutils.fs.ls(f"{FILES}/{src}")}
in_table = {r[0].isoformat() for r in
            spark.table(f"workspace.bronze.{src}").select("_partition_day").distinct().collect()}
missing = sorted(on_disk - in_table)
print("missing days:", missing)

for day in missing:
    for f in dbutils.fs.ls(f"{FILES}/{src}/{day}"):
        if not f.name.endswith(".meta.json"):
            print(day, f.name, f.size, "bytes:", dbutils.fs.head(f.path, 200))

missing days: ['2025-02-09', '2026-01-09', '2026-03-05']
2025-02-09 20261013T232554330473Z_b0fb7888.json 44 bytes: {"metadata":{"datasets":["DGWS"]},"data":[]}
2026-01-09 20261013T232841935250Z_b0fb7888.json 44 bytes: {"metadata":{"datasets":["DGWS"]},"data":[]}
2026-03-05 20261013T232909440382Z_b0fb7888.json 44 bytes: {"metadata":{"datasets":["DGWS"]},"data":[]}


In [0]:
files = (
    spark.read.option("wholetext", True)
    .text(f"{FILES}/*/*/*Z_????????.json")
    .select(F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_size").alias("bytes"),
            "value")
    .withColumn("source", F.regexp_extract("_source_file", r"/files/([^/]+)/", 1))
    .withColumn("_partition_day",
                F.to_date(F.regexp_extract("_source_file", r"/(\d{4}-\d{2}-\d{2})/", 1)))
    .withColumn("_fetched_at",
                F.to_timestamp(F.regexp_extract("_source_file", r"/(\d{8}T\d{12})Z_", 1),
                               "yyyyMMdd'T'HHmmssSSSSSS"))
    .withColumn("record_count",
                F.when(F.col("source") == "open_meteo_ecmwf",
                       F.coalesce(F.expr("json_array_length(value)"), F.lit(1)))
                 .otherwise(F.expr("json_array_length(get_json_object(value, '$.data'))")))
    .withColumn("sha_in_name", F.regexp_extract("_source_file", r"Z_([0-9a-f]{8})\.json$", 1))
    .withColumn("sha_ok", F.substring(F.sha2(F.col("value").cast("binary"), 256), 1, 8)
                          == F.col("sha_in_name"))
    .withColumn("_ingested_at", F.current_timestamp())
    .drop("value")
)

(files.write.format("delta").mode("overwrite").option("overwriteSchema", True)
      .saveAsTable("workspace.bronze.landing_files"))

display(spark.sql("""
    SELECT source,
           COUNT(*)                                   AS files,
           COUNT(DISTINCT _partition_day)             AS days,
           SUM(CASE WHEN record_count = 0 THEN 1 END) AS empty_files,
           SUM(CASE WHEN NOT sha_ok THEN 1 END)       AS bad_fingerprints
    FROM workspace.bronze.landing_files
    GROUP BY source ORDER BY source
"""))

source,files,days,empty_files,bad_fingerprints
carbon_intensity,743,743,null,null
elexon_demand_forecast,2233,744,null,null
elexon_demand_outturn,743,743,null,null
elexon_generation_by_fuel,743,743,null,null
elexon_system_prices,743,743,null,null
elexon_wind_forecast,744,744,null,null
elexon_wind_solar_actual,743,743,null,null
elexon_wind_solar_day_ahead,741,741,3,null
open_meteo_ecmwf,740,740,null,null


In [0]:
from pyspark.sql import functions as F
FILES = "/Volumes/workspace/bronze/landing/files"
OM_SOURCE = "open_meteo_ecmwf"
# Same order as config/settings.yaml, which is the order the API returns them
LOCATIONS = ["wind_off_hornsea", "wind_off_dogger", "wind_off_eastanglia",
             "wind_off_seagreen", "wind_off_moray", "wind_on_scotland",
             "solar_cambs", "solar_kent", "solar_somerset",
             "temp_london", "temp_birmingham", "temp_manchester"]
VARS = ["temperature_2m", "shortwave_radiation", "wind_speed_100m", "cloud_cover"]

om_schema = ("array<struct<latitude:double, longitude:double, elevation:double, "
             "hourly:struct<time:array<string>, "
             + ", ".join(f"{v}:array<double>" for v in VARS) + ">>>")

text = (spark.read.option("wholetext", True)
        .text(f"{FILES}/{OM_SOURCE}/*/*Z_????????.json")
        .select("value", F.col("_metadata.file_path").alias("_source_file")))

# A one location reply is a single object, not a list; wrap it so both look the same
as_list = F.when(F.trim("value").startswith("{"), F.concat(F.lit("["), "value", F.lit("]"))) \
           .otherwise(F.col("value"))

locs = (text
        .select(F.posexplode(F.from_json(as_list, om_schema)).alias("_location_index", "loc"),
                "_source_file"))

hours = (locs
         .select("_location_index", "loc", "_source_file",
                 F.posexplode("loc.hourly.time").alias("h", "time"))
         .select(
             F.element_at(F.array(*[F.lit(n) for n in LOCATIONS]),
                          F.col("_location_index") + 1).alias("location"),
             "_location_index",
             F.col("loc.latitude").alias("latitude"),
             F.col("loc.longitude").alias("longitude"),
             F.col("loc.elevation").alias("elevation"),
             "time",
             *[F.col(f"loc.hourly.{v}")[F.col("h")].alias(v) for v in VARS],
             "_source_file")
         .withColumn("_partition_day",
                     F.to_date(F.regexp_extract("_source_file", r"/(\d{4}-\d{2}-\d{2})/", 1)))
         .withColumn("_fetched_at",
                     F.to_timestamp(F.regexp_extract("_source_file", r"/(\d{8}T\d{12})Z_", 1),
                                    "yyyyMMdd'T'HHmmssSSSSSS"))
         .withColumn("_ingested_at", F.current_timestamp()))

(hours.write.format("delta").mode("overwrite").option("overwriteSchema", True)
      .saveAsTable(f"workspace.bronze.{OM_SOURCE}"))

t = spark.table(f"workspace.bronze.{OM_SOURCE}")
print("rows:", t.count(), "| files:", t.select("_source_file").distinct().count(),
      "| days:", t.select("_partition_day").distinct().count())
display(t.groupBy("_source_file").agg(F.countDistinct("location").alias("locations"),
                                      F.count("*").alias("rows"))
         .groupBy("locations", "rows").count().orderBy("rows"))
display(t.filter("_partition_day = '2025-10-26' AND location = 'temp_london'")
         .orderBy("time").limit(6))

rows: 1491840 | files: 740 | days: 740


locations,rows,count
12,2016,740


location,_location_index,latitude,longitude,elevation,time,temperature_2m,shortwave_radiation,wind_speed_100m,cloud_cover,_source_file,_partition_day,_fetched_at,_ingested_at
temp_london,9,51.493847,-0.1630249,17.0,2025-10-25T00:00,8.7,null,22.5,100.0,dbfs:/Volumes/workspace/bronze/landing/files/open_meteo_ecmwf/2025-10-26/20261014T001219361500Z_a6efae95.json,2025-10-26,2026-10-14T00:12:19.361Z,2026-10-16T02:25:43.056Z
temp_london,9,51.493847,-0.1630249,17.0,2025-10-25T01:00,8.8,0.0,23.1,88.0,dbfs:/Volumes/workspace/bronze/landing/files/open_meteo_ecmwf/2025-10-26/20261014T001219361500Z_a6efae95.json,2025-10-26,2026-10-14T00:12:19.361Z,2026-10-16T02:25:43.056Z
temp_london,9,51.493847,-0.1630249,17.0,2025-10-25T02:00,8.4,0.0,26.7,65.0,dbfs:/Volumes/workspace/bronze/landing/files/open_meteo_ecmwf/2025-10-26/20261014T001219361500Z_a6efae95.json,2025-10-26,2026-10-14T00:12:19.361Z,2026-10-16T02:25:43.056Z
temp_london,9,51.493847,-0.1630249,17.0,2025-10-25T03:00,8.2,0.0,25.2,94.0,dbfs:/Volumes/workspace/bronze/landing/files/open_meteo_ecmwf/2025-10-26/20261014T001219361500Z_a6efae95.json,2025-10-26,2026-10-14T00:12:19.361Z,2026-10-16T02:25:43.056Z
temp_london,9,51.493847,-0.1630249,17.0,2025-10-25T04:00,8.6,0.0,28.8,42.0,dbfs:/Volumes/workspace/bronze/landing/files/open_meteo_ecmwf/2025-10-26/20261014T001219361500Z_a6efae95.json,2025-10-26,2026-10-14T00:12:19.361Z,2026-10-16T02:25:43.056Z
temp_london,9,51.493847,-0.1630249,17.0,2025-10-25T05:00,8.7,0.0,29.3,95.0,dbfs:/Volumes/workspace/bronze/landing/files/open_meteo_ecmwf/2025-10-26/20261014T001219361500Z_a6efae95.json,2025-10-26,2026-10-14T00:12:19.361Z,2026-10-16T02:25:43.056Z


In [0]:
from pyspark.sql import functions as F

FILES = "/Volumes/workspace/bronze/landing/files"
OM_SOURCE = "open_meteo_ecmwf"
# Same order as config/settings.yaml, which is the order the API returns them
LOCATIONS = ["wind_off_hornsea", "wind_off_dogger", "wind_off_eastanglia",
             "wind_off_seagreen", "wind_off_moray", "wind_on_scotland",
             "solar_cambs", "solar_kent", "solar_somerset",
             "temp_london", "temp_birmingham", "temp_manchester"]
VARS = ["temperature_2m", "shortwave_radiation", "wind_speed_100m", "cloud_cover"]

om_schema = ("array<struct<latitude:double, longitude:double, elevation:double, "
             "hourly:struct<time:array<string>, "
             + ", ".join(f"{v}:array<double>" for v in VARS) + ">>>")

text = (spark.read.option("wholetext", True)
        .text(f"{FILES}/{OM_SOURCE}/*/*Z_????????.json")
        .select("value", F.col("_metadata.file_path").alias("_source_file")))

as_list = (F.when(F.trim("value").startswith("{"), F.concat(F.lit("["), "value", F.lit("]")))
            .otherwise(F.col("value")))

locs = text.select(F.posexplode(F.from_json(as_list, om_schema)).alias("_location_index", "loc"),
                   "_source_file")

hours = (locs
         .select("_location_index", "loc", "_source_file",
                 F.posexplode("loc.hourly.time").alias("h", "time"))
         .select(
             F.element_at(F.array(*[F.lit(n) for n in LOCATIONS]),
                          F.col("_location_index") + 1).alias("location"),
             "_location_index",
             F.col("loc.latitude").alias("latitude"),
             F.col("loc.longitude").alias("longitude"),
             F.col("loc.elevation").alias("elevation"),
             "time",
             *[F.col(f"loc.hourly.{v}")[F.col("h")].alias(v) for v in VARS],
             "_source_file")
         .withColumn("_partition_day",
                     F.to_date(F.regexp_extract("_source_file", r"/(\d{4}-\d{2}-\d{2})/", 1)))
         .withColumn("_fetched_at",
                     F.to_timestamp(F.regexp_extract("_source_file", r"/(\d{8}T\d{12})Z_", 1),
                                    "yyyyMMdd'T'HHmmssSSSSSS"))
         .withColumn("_ingested_at", F.current_timestamp()))

# NEW: which forecast run each file really holds (normally 00:00 the day before; older after a fallback)
meta = (spark.read.option("multiLine", True)
        .json(f"{FILES}/{OM_SOURCE}/*/*.meta.json")
        .select(F.regexp_replace(F.col("_metadata.file_path"), r"\.meta\.json$", ".json")
                 .alias("_source_file"),
                F.to_timestamp("params.run", "yyyy-MM-dd'T'HH:mm").alias("run_time_utc")))
hours = hours.join(meta, "_source_file", "left")

(hours.write.format("delta").mode("overwrite").option("overwriteSchema", True)
      .saveAsTable(f"workspace.bronze.{OM_SOURCE}"))

t = spark.table(f"workspace.bronze.{OM_SOURCE}")
print("rows:", t.count(), "| files:", t.select("_source_file").distinct().count(),
      "| days:", t.select("_partition_day").distinct().count(),
      "| rows without run time:", t.filter("run_time_utc IS NULL").count())
display(t.groupBy(F.hour("run_time_utc").alias("run_hour_utc"))
         .agg(F.countDistinct("_partition_day").alias("days"))
         .orderBy("run_hour_utc"))

rows: 1492704 | files: 742 | days: 742 | rows without run time: 0


run_hour_utc,days
0,738
6,3
12,1


In [0]:
from pyspark.sql import functions as F

FILES = "/Volumes/workspace/bronze/landing/files"
source = "elexon_settled_prices"

raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{FILES}/{source}/*/*Z_????????.json")
    .select(
        F.explode("data").alias("r"),
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .select("r.*", "_source_file")
    .withColumn("_partition_day",
                F.to_date(F.regexp_extract("_source_file", r"/(\d{4}-\d{2}-\d{2})/", 1)))
    .withColumn("_fetched_at",
                F.to_timestamp(F.regexp_extract("_source_file", r"/(\d{8}T\d{12})Z_", 1),
                               "yyyyMMdd'T'HHmmssSSSSSS"))
    .withColumn("_ingested_at", F.current_timestamp())
)

raw.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.bronze.{source}")

t = spark.table(f"workspace.bronze.{source}")
t.printSchema()
print("rows:", t.count(), "| days:", t.select("_partition_day").distinct().count(),
      "| files:", t.select("_source_file").distinct().count())
print("rows where file day != settlement date:",
      t.filter(F.col("_partition_day") != F.to_date("settlementDate")).count())
display(t.groupBy("runType").count().orderBy("runType"))

root
 |-- runType: string (nullable = true)
 |-- settlementDate: string (nullable = true)
 |-- settlementPeriod: long (nullable = true)
 |-- systemBuyPrice: double (nullable = true)
 |-- systemSellPrice: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _partition_day: date (nullable = true)
 |-- _fetched_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

rows: 36143 | days: 753 | files: 753
rows where file day != settlement date: 0


runType,count
BM,239
II,816
R1,3168
R2,4894
R3,9554
RF,16272
SF,1200


In [0]:
from pyspark.sql import functions as F

FILES = "/Volumes/workspace/bronze/landing/files"
source = "elexon_settled_prices"

raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{FILES}/{source}/*/*Z_????????.json")
    .select(
        F.explode("data").alias("r"),
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .select("r.*", "_source_file")
    .withColumn("_partition_day",
                F.to_date(F.regexp_extract("_source_file", r"/(\d{4}-\d{2}-\d{2})/", 1)))
    .withColumn("_fetched_at",
                F.to_timestamp(F.regexp_extract("_source_file", r"/(\d{8}T\d{12})Z_", 1),
                               "yyyyMMdd'T'HHmmssSSSSSS"))
)

raw.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.bronze.{source}")

t = spark.table(f"workspace.bronze.{source}")
t.printSchema()
print("rows:", t.count(), "| days:", t.select("_partition_day").distinct().count(),
      "| files:", t.select("_source_file").distinct().count())
print("rows where file day != settlement date:",
      t.filter(F.col("_partition_day") != F.to_date("settlementDate")).count())
display(t.groupBy("runType").count().orderBy("runType"))

root
 |-- runType: string (nullable = true)
 |-- settlementDate: string (nullable = true)
 |-- settlementPeriod: long (nullable = true)
 |-- systemBuyPrice: double (nullable = true)
 |-- systemSellPrice: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _partition_day: date (nullable = true)
 |-- _fetched_at: timestamp (nullable = true)

rows: 36143 | days: 753 | files: 753
rows where file day != settlement date: 0


runType,count
BM,239
II,816
R1,3168
R2,4894
R3,9554
RF,16272
SF,1200


In [0]:
from pyspark.sql import functions as F

FILES = "/Volumes/workspace/bronze/landing/files"

files = (
    spark.read.option("wholetext", True)
    .text(f"{FILES}/*/*/*Z_????????.json")
    .select(F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_size").alias("bytes"),
            "value")
    .withColumn("source", F.regexp_extract("_source_file", r"/files/([^/]+)/", 1))
    .withColumn("_partition_day",
                F.to_date(F.regexp_extract("_source_file", r"/(\d{4}-\d{2}-\d{2})/", 1)))
    .withColumn("_fetched_at",
                F.to_timestamp(F.regexp_extract("_source_file", r"/(\d{8}T\d{12})Z_", 1),
                               "yyyyMMdd'T'HHmmssSSSSSS"))
    .withColumn("record_count",
                F.when(F.col("source") == "open_meteo_ecmwf",
                       F.coalesce(F.expr("json_array_length(value)"), F.lit(1)))
                 .otherwise(F.expr("json_array_length(get_json_object(value, '$.data'))")))
    .withColumn("sha_in_name", F.regexp_extract("_source_file", r"Z_([0-9a-f]{8})\.json$", 1))
    .withColumn("sha_ok", F.substring(F.sha2(F.col("value").cast("binary"), 256), 1, 8)
                          == F.col("sha_in_name"))
    .withColumn("_ingested_at", F.current_timestamp())
    .drop("value")
)

(files.write.format("delta").mode("overwrite").option("overwriteSchema", True)
      .saveAsTable("workspace.bronze.landing_files"))

display(spark.sql("""
    SELECT COALESCE(source, 'TOTAL') AS source,
           COUNT(*) AS files,
           COUNT(DISTINCT _partition_day) AS days,
           COALESCE(SUM(CASE WHEN record_count = 0 THEN 1 END), 0) AS empty_files,
           COALESCE(SUM(CASE WHEN NOT sha_ok THEN 1 END), 0) AS bad_fingerprints,
           ROUND(SUM(bytes) / 1024 / 1024, 1) AS mb
    FROM workspace.bronze.landing_files
    GROUP BY ROLLUP(source)
    ORDER BY source = 'TOTAL', source
"""))

source,files,days,empty_files,bad_fingerprints,mb
carbon_intensity,743,743,0,0,3.9
elexon_demand_forecast,2233,744,0,0,17.6
elexon_demand_outturn,743,743,0,0,6.9
elexon_generation_by_fuel,743,743,0,0,123.3
elexon_settled_prices,753,753,0,0,4.9
elexon_system_prices,743,743,0,0,30.2
elexon_wind_forecast,744,744,0,0,7.4
elexon_wind_solar_actual,743,743,0,0,21.0
elexon_wind_solar_day_ahead,741,741,3,0,23.1
open_meteo_ecmwf,742,742,0,0,55.5
